# 02 Validation

This notebook runs read-only validation checks against the local DuckDB warehouse.

It focuses on table existence, duplicate identifiers, unresolved links, team dimension ambiguity, and source-position overlap.

In [ ]:
from pathlib import Path
import sys
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
import duckdb
import pandas as pd
from IPython.display import Markdown, display

In [ ]:
DB_PATH = ROOT / "data" / "nba_analytics.duckdb"
SQL_PATH = ROOT / "sql" / "validation_checks_duckdb.sql"

if not DB_PATH.exists():
    raise FileNotFoundError(f"{DB_PATH} does not exist. Run 01_build_database.ipynb first.")

con = duckdb.connect(str(DB_PATH))
con.execute("SELECT current_database() AS database_name").fetchdf()

In [ ]:
raw_sql = SQL_PATH.read_text()
parts = [part.strip() for part in re.split(r"\n(?=-- \d+\.)", raw_sql.strip()) if part.strip()]

queries = []
for part in parts:
    lines = part.splitlines()
    if not lines or not re.match(r"^-- \d+\.", lines[0]):
        continue
    title = lines[0].removeprefix("-- ").strip()
    sql = "\n".join(line for line in lines[1:] if not line.strip().startswith("--")).strip()
    queries.append((title, sql))

pd.DataFrame({"validation_check": [title for title, _ in queries]})

In [ ]:
results = {}

for title, sql in queries:
    display(Markdown(f"### {title}"))
    df = con.execute(sql).fetchdf()
    results[title] = df
    display(df)

## Reading The Results

- duplicate `gameId` rows signal that game-level deduplication still needs cleanup
- non-zero missing key counts indicate unresolved fact-to-dimension links
- multiple rows for the same `teamName` reflect historical naming and city variation
- source-position overlap means raw Guard/Forward/Center benchmarking is weak in this dataset

The analysis notebook avoids that issue by using historical player-season roles inferred from box-score production.

The current results are summarized in `docs/VALIDATION_RESULTS.md`.